In [1]:
import json
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

WIKIR_DIR = Path("/Users/uni/homework13April/wikIR1k")
MIRAGE_DIR = Path("/Users/uni/homework13April/mirage")
EMB_DIR    = Path("/Users/uni/homework13April/embeddings")

print("Imports OK")

Using device: mps
Imports OK


In [2]:
# Load WikIR corpus
docs_df     = pd.read_csv(WIKIR_DIR / "documents.csv")
doc_ids     = docs_df["id_right"].tolist()
doc_texts   = docs_df["text_right"].tolist()
doc_id_to_idx = {doc_id: idx for idx, doc_id in enumerate(doc_ids)}

# Load test queries
test_queries_df = pd.read_csv(WIKIR_DIR / "test" / "queries.csv")
test_qids   = test_queries_df["id_left"].tolist()
test_qtexts = test_queries_df["text_left"].tolist()

# Load qrels
def load_qrels(path):
    qrels = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            qid, _, docid, rel = int(parts[0]), parts[1], int(parts[2]), int(parts[3])
            if qid not in qrels:
                qrels[qid] = {}
            qrels[qid][docid] = rel
    return qrels

test_qrels = load_qrels(WIKIR_DIR / "test" / "qrels")

# Load BM25 results (TREC format: qid Q0 docid rank score tag)
def load_bm25_res(path, top_k=100):
    results = defaultdict(list)
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            qid, docid, score = int(parts[0]), int(parts[2]), float(parts[4])
            results[qid].append((docid, score))
    return {qid: docs[:top_k] for qid, docs in results.items()}

bm25_test = load_bm25_res(WIKIR_DIR / "test" / "BM25.res")

print(f"Corpus: {len(doc_ids):,} docs")
print(f"Test queries: {len(test_qids)}")
print(f"BM25 queries: {len(bm25_test)}")

Corpus: 369,721 docs
Test queries: 100
BM25 queries: 100


In [3]:
# Load MIRAGE data
with open(MIRAGE_DIR / "dataset.json") as f:
    mirage_dataset = json.load(f)
with open(MIRAGE_DIR / "doc_pool.json") as f:
    mirage_doc_pool = json.load(f)
with open(MIRAGE_DIR / "oracle.json") as f:
    mirage_oracle = json.load(f)

mirage_docs_by_query = defaultdict(list)
for doc in mirage_doc_pool:
    mirage_docs_by_query[doc["mapped_id"]].append(doc)

print(f"MIRAGE queries: {len(mirage_dataset)}")
print(f"MIRAGE doc pool: {len(mirage_doc_pool)}")

MIRAGE queries: 7560
MIRAGE doc pool: 37800


In [4]:
# Evaluation functions
def precision_at_k(ranked_doc_ids, relevant_docs, k):
    top_k = ranked_doc_ids[:k]
    hits = sum(1 for d in top_k if d in relevant_docs and relevant_docs[d] > 0)
    return hits / k

def ap_at_k(ranked_doc_ids, relevant_docs, k):
    hits, score = 0, 0.0
    for i, doc in enumerate(ranked_doc_ids[:k]):
        if doc in relevant_docs and relevant_docs[doc] > 0:
            hits += 1
            score += hits / (i + 1)
    num_rel = sum(1 for r in relevant_docs.values() if r > 0)
    return score / min(num_rel, k) if num_rel > 0 else 0.0

def dcg_at_k(ranked_doc_ids, relevant_docs, k):
    score = 0.0
    for i, doc in enumerate(ranked_doc_ids[:k]):
        rel = relevant_docs.get(doc, 0)
        score += rel / np.log2(i + 2)
    return score

def ndcg_at_k(ranked_doc_ids, relevant_docs, k):
    actual = dcg_at_k(ranked_doc_ids, relevant_docs, k)
    ideal_rels = sorted(relevant_docs.values(), reverse=True)[:k]
    ideal = sum(r / np.log2(i + 2) for i, r in enumerate(ideal_rels))
    return actual / ideal if ideal > 0 else 0.0

def evaluate(results_per_query, qrels):
    p1, p10, p20, map20, ndcg20 = [], [], [], [], []
    for qid, ranked in results_per_query.items():
        rel = qrels.get(qid, {})
        p1.append(precision_at_k(ranked, rel, 1))
        p10.append(precision_at_k(ranked, rel, 10))
        p20.append(precision_at_k(ranked, rel, 20))
        map20.append(ap_at_k(ranked, rel, 20))
        ndcg20.append(ndcg_at_k(ranked, rel, 20))
    return {
        "P@1":     round(np.mean(p1), 4),
        "P@10":    round(np.mean(p10), 4),
        "P@20":    round(np.mean(p20), 4),
        "MAP@20":  round(np.mean(map20), 4),
        "nDCG@20": round(np.mean(ndcg20), 4),
    }

print("Evaluation functions ready")

Evaluation functions ready


In [5]:
def wikir_rerank(model, model_name, doc_embs_cache, doc_id_to_idx,
                 bm25_results, query_ids, query_texts, device, k):
    """Re-rank top-k BM25 results for WikIR using cached doc embeddings."""
    t_start = time.time()

    query_embs = model.encode(
        query_texts, normalize_embeddings=True,
        convert_to_numpy=True, device=device, show_progress_bar=False
    )

    results = {}
    for i, qid in enumerate(query_ids):
        top_k_docs = bm25_results.get(qid, [])[:k]
        valid = [(docid, score) for docid, score in top_k_docs if docid in doc_id_to_idx]
        if not valid:
            results[qid] = [docid for docid, _ in top_k_docs]
            continue
        indices = [doc_id_to_idx[docid] for docid, _ in valid]
        cand_embs = doc_embs_cache[indices]
        scores = cand_embs @ query_embs[i]
        ranked_order = np.argsort(-scores)
        results[qid] = [valid[j][0] for j in ranked_order]

    elapsed = time.time() - t_start
    return results, elapsed


def mirage_rerank(model, model_name, mirage_dataset, mirage_docs_by_query,
                  mirage_oracle, device):
    """BM25 + neural re-ranking for MIRAGE. Deduplicates by doc_name."""
    results = {}
    qrels   = {}
    t_start = time.time()

    for entry in tqdm(mirage_dataset, desc=f"MIRAGE rerank {model_name.split('/')[-1]}"):
        qid   = entry["query_id"]
        query = entry["query"]
        candidate_docs = mirage_docs_by_query[qid]
        if not candidate_docs:
            continue

        doc_chunks = [d["doc_chunk"] for d in candidate_docs]
        doc_names  = [d["doc_name"]  for d in candidate_docs]

        # BM25 score each candidate
        tokenized_corpus = [chunk.lower().split() for chunk in doc_chunks]
        tokenized_query  = query.lower().split()
        bm25 = BM25Okapi(tokenized_corpus)
        bm25_scores = bm25.get_scores(tokenized_query)

        # Neural re-rank: encode query + candidates
        query_emb = model.encode(query, normalize_embeddings=True, convert_to_numpy=True)
        doc_embs  = model.encode(doc_chunks, normalize_embeddings=True, convert_to_numpy=True)
        neural_scores = doc_embs @ query_emb

        # Re-rank by neural score (BM25 was used for candidate selection)
        ranked_indices = np.argsort(-neural_scores)

        # Deduplicate by doc_name
        seen = set()
        ranked_names = []
        for idx in ranked_indices:
            name = doc_names[idx]
            if name not in seen:
                seen.add(name)
                ranked_names.append(name)

        results[qid] = ranked_names

        relevant_name = mirage_oracle[qid]["doc_name"] if qid in mirage_oracle else None
        unique_names  = list(dict.fromkeys(doc_names))
        qrels[qid]    = {name: (1 if name == relevant_name else 0) for name in unique_names}

    elapsed = time.time() - t_start
    return results, qrels, elapsed

print("Re-ranking functions ready")

Re-ranking functions ready


In [6]:
## Model 1: multi-qa-MiniLM-L6-cos-v1
MODEL1_NAME = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
print(f"Loading {MODEL1_NAME}...")
model1 = SentenceTransformer(MODEL1_NAME, device=device)
safe1  = MODEL1_NAME.replace("/", "_")
doc_embs_m1 = np.load(EMB_DIR / f"wikir_docs_{safe1}.npy")
print(f"Loaded cached embeddings: {doc_embs_m1.shape}")

Loading sentence-transformers/multi-qa-MiniLM-L6-cos-v1...
Loaded cached embeddings: (369721, 384)


In [7]:
# WikIR re-ranking — Model 1 — different k values
K_VALUES = [10, 20, 50, 100]
wikir_m1_results = {}

for k in K_VALUES:
    res, elapsed = wikir_rerank(
        model1, MODEL1_NAME, doc_embs_m1, doc_id_to_idx,
        bm25_test, test_qids, test_qtexts, device, k
    )
    metrics = evaluate(res, test_qrels)
    metrics["time_s"] = round(elapsed, 2)
    metrics["ms_per_query"] = round(elapsed / len(test_qids) * 1000, 1)
    wikir_m1_results[k] = metrics
    print(f"k={k:3d}: {metrics}")

k= 10: {'P@1': 0.43, 'P@10': 0.127, 'P@20': 0.0635, 'MAP@20': 0.1064, 'nDCG@20': 0.2259, 'time_s': 0.48, 'ms_per_query': 4.8}
k= 20: {'P@1': 0.44, 'P@10': 0.146, 'P@20': 0.0945, 'MAP@20': 0.1279, 'nDCG@20': 0.268, 'time_s': 0.06, 'ms_per_query': 0.6}
k= 50: {'P@1': 0.47, 'P@10': 0.155, 'P@20': 0.1045, 'MAP@20': 0.1342, 'nDCG@20': 0.2942, 'time_s': 0.05, 'ms_per_query': 0.5}
k=100: {'P@1': 0.47, 'P@10': 0.148, 'P@20': 0.1045, 'MAP@20': 0.1325, 'nDCG@20': 0.299, 'time_s': 0.11, 'ms_per_query': 1.1}


In [8]:
# MIRAGE re-ranking — Model 1
mirage_m1_results, mirage_qrels, mirage_m1_elapsed = mirage_rerank(
    model1, MODEL1_NAME, mirage_dataset, mirage_docs_by_query, mirage_oracle, device
)
mirage_m1_metrics = evaluate(mirage_m1_results, mirage_qrels)
mirage_m1_metrics["time_s"] = round(mirage_m1_elapsed, 2)
mirage_m1_metrics["ms_per_query"] = round(mirage_m1_elapsed / len(mirage_dataset) * 1000, 1)
print(f"MIRAGE — {MODEL1_NAME.split('/')[-1]}: {mirage_m1_metrics}")

MIRAGE rerank multi-qa-MiniLM-L6-cos-v1: 100%|██████████| 7560/7560 [07:37<00:00, 16.52it/s]


MIRAGE — multi-qa-MiniLM-L6-cos-v1: {'P@1': 0.8774, 'P@10': 0.1, 'P@20': 0.05, 'MAP@20': 0.9331, 'nDCG@20': 0.9503, 'time_s': 457.57, 'ms_per_query': 60.5}


In [9]:
import gc
del model1, doc_embs_m1
gc.collect()
print("model1 freed")

model1 freed


In [10]:
## Model 2: msmarco-distilbert-dot-v5
MODEL2_NAME = "sentence-transformers/msmarco-distilbert-dot-v5"
print(f"Loading {MODEL2_NAME}...")
model2 = SentenceTransformer(MODEL2_NAME, device=device)
safe2  = MODEL2_NAME.replace("/", "_")
doc_embs_m2 = np.load(EMB_DIR / f"wikir_docs_{safe2}.npy")
print(f"Loaded cached embeddings: {doc_embs_m2.shape}")

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 6ebab1ce-247d-4a25-b8bd-d038f9173c09)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/msmarco-distilbert-dot-v5/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Loading sentence-transformers/msmarco-distilbert-dot-v5...
Loaded cached embeddings: (369721, 768)


In [11]:
# WikIR re-ranking — Model 2 — different k values
wikir_m2_results = {}

for k in K_VALUES:
    res, elapsed = wikir_rerank(
        model2, MODEL2_NAME, doc_embs_m2, doc_id_to_idx,
        bm25_test, test_qids, test_qtexts, device, k
    )
    metrics = evaluate(res, test_qrels)
    metrics["time_s"] = round(elapsed, 2)
    metrics["ms_per_query"] = round(elapsed / len(test_qids) * 1000, 1)
    wikir_m2_results[k] = metrics
    print(f"k={k:3d}: {metrics}")

k= 10: {'P@1': 0.45, 'P@10': 0.127, 'P@20': 0.0635, 'MAP@20': 0.1036, 'nDCG@20': 0.2282, 'time_s': 0.62, 'ms_per_query': 6.2}
k= 20: {'P@1': 0.5, 'P@10': 0.144, 'P@20': 0.0945, 'MAP@20': 0.1298, 'nDCG@20': 0.2769, 'time_s': 0.07, 'ms_per_query': 0.7}
k= 50: {'P@1': 0.57, 'P@10': 0.144, 'P@20': 0.1005, 'MAP@20': 0.1338, 'nDCG@20': 0.3018, 'time_s': 0.06, 'ms_per_query': 0.6}
k=100: {'P@1': 0.58, 'P@10': 0.148, 'P@20': 0.0995, 'MAP@20': 0.1355, 'nDCG@20': 0.3086, 'time_s': 0.09, 'ms_per_query': 0.9}


In [12]:
# MIRAGE re-ranking — Model 2
mirage_m2_results, _, mirage_m2_elapsed = mirage_rerank(
    model2, MODEL2_NAME, mirage_dataset, mirage_docs_by_query, mirage_oracle, device
)
mirage_m2_metrics = evaluate(mirage_m2_results, mirage_qrels)
mirage_m2_metrics["time_s"] = round(mirage_m2_elapsed, 2)
mirage_m2_metrics["ms_per_query"] = round(mirage_m2_elapsed / len(mirage_dataset) * 1000, 1)
print(f"MIRAGE — {MODEL2_NAME.split('/')[-1]}: {mirage_m2_metrics}")

MIRAGE rerank msmarco-distilbert-dot-v5: 100%|██████████| 7560/7560 [10:03<00:00, 12.52it/s]

MIRAGE — msmarco-distilbert-dot-v5: {'P@1': 0.8619, 'P@10': 0.1, 'P@20': 0.05, 'MAP@20': 0.924, 'nDCG@20': 0.9435, 'time_s': 603.76, 'ms_per_query': 79.9}


In [13]:
# Results table — WikIR re-ranking across k values
rows = []
for k in K_VALUES:
    m1 = wikir_m1_results[k]
    m2 = wikir_m2_results[k]
    rows.append({"Model": "multi-qa-MiniLM-L6-cos-v1", "Dataset": "WikIR", "k": k,
                 **{kk: vv for kk, vv in m1.items()}})
    rows.append({"Model": "msmarco-distilbert-dot-v5",  "Dataset": "WikIR", "k": k,
                 **{kk: vv for kk, vv in m2.items()}})

# MIRAGE (k=5 max, all candidates used)
rows.append({"Model": "multi-qa-MiniLM-L6-cos-v1", "Dataset": "MIRAGE", "k": 5, **mirage_m1_metrics})
rows.append({"Model": "msmarco-distilbert-dot-v5",  "Dataset": "MIRAGE", "k": 5, **mirage_m2_metrics})

results_df = pd.DataFrame(rows)
print("=== Task 2 — Re-ranking Results ===")
print(results_df.to_string(index=False))

=== Task 2 — Re-ranking Results ===
                    Model Dataset   k    P@1  P@10   P@20  MAP@20  nDCG@20  time_s  ms_per_query
multi-qa-MiniLM-L6-cos-v1   WikIR  10 0.4300 0.127 0.0635  0.1064   0.2259    0.48           4.8
msmarco-distilbert-dot-v5   WikIR  10 0.4500 0.127 0.0635  0.1036   0.2282    0.62           6.2
multi-qa-MiniLM-L6-cos-v1   WikIR  20 0.4400 0.146 0.0945  0.1279   0.2680    0.06           0.6
msmarco-distilbert-dot-v5   WikIR  20 0.5000 0.144 0.0945  0.1298   0.2769    0.07           0.7
multi-qa-MiniLM-L6-cos-v1   WikIR  50 0.4700 0.155 0.1045  0.1342   0.2942    0.05           0.5
msmarco-distilbert-dot-v5   WikIR  50 0.5700 0.144 0.1005  0.1338   0.3018    0.06           0.6
multi-qa-MiniLM-L6-cos-v1   WikIR 100 0.4700 0.148 0.1045  0.1325   0.2990    0.11           1.1
msmarco-distilbert-dot-v5   WikIR 100 0.5800 0.148 0.0995  0.1355   0.3086    0.09           0.9
multi-qa-MiniLM-L6-cos-v1  MIRAGE   5 0.8774 0.100 0.0500  0.9331   0.9503  457.57         